In [1]:
import camelot
import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
votes_2022_df=pd.read_csv('../data/voter_numbers_2022_with_alt_map.csv')
votes_2022_df.head()

,county_id,NAME10,dem_votes,rep_votes,indep_votes,og_district,alt_district
0,0,Adair,973,2166,NaN,3,4
1,1,Adams,510,1126,NaN,3,3
2,2,Allamakee,1932,3820,NaN,2,2
3,3,Appanoose,1412,3249,NaN,3,3
4,4,Audubon,633,1639,51.0,4,4


In [3]:
raw_data=camelot.read_pdf('../data/canvsummary_2024.pdf', pages='2-14')
raw_data

<TableList n=13>

In [4]:
raw_data[12].df.tail(-1).info()
raw_data[0].df.tail(-1)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 1 to 24
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       24 non-null     object
 1   1       24 non-null     object
 2   2       24 non-null     object
 3   3       24 non-null     object
 4   4       24 non-null     object
 5   5       24 non-null     object
 6   6       24 non-null     object
 7   7       24 non-null     object
 8   8       24 non-null     object
 9   9       24 non-null     object
 10  10      24 non-null     object
 11  11      24 non-null     object
 12  12      24 non-null     object
dtypes: object(13)
memory usage: 2.6+ KB


,0,1,2,3,4,5,6,7,8,9,10,11,12
1,Adair,Election \nDay,558,"1,915",14,4,0,2,21,8,14,1,"2,537"
2,,Absentee,528,"1,001",6,1,1,0,13,8,8,1,"1,567"
3,,Total,"1,086","2,916",20,5,1,2,34,16,22,2,"4,104"
4,Adams,Election \nDay,266,878,5,1,0,0,13,4,6,1,"1,174"
5,,Absentee,310,639,4,0,0,0,15,0,6,5,979
6,,Total,576,"1,517",9,1,0,0,28,4,12,6,"2,153"
7,Allamakee,Election \nDay,"1,070","2,893",23,3,6,0,27,11,39,3,"4,075"
8,,Absentee,"1,280","1,964",5,2,1,1,28,4,40,10,"3,335"
9,,Total,"2,350","4,857",28,5,7,1,55,15,79,13,"7,410"
10,Appanoose,Election \nDay,787,"2,985",18,0,3,1,28,11,15,0,"3,848"


In [5]:
voter_data_2024=pd.concat([raw_data[i].df.tail(-1) for i in range(13)],ignore_index=True)
voter_data_2024.tail(5)


,0,1,2,3,4,5,6,7,8,9,10,11,12
296,,Total,"1,508","2,715",13,0,3,1,38,9,21,5,"4,313"
297,Wright,Election \nDay,"1,045","2,505",7,1,5,3,47,11,20,2,"3,646"
298,,Absentee,725,"1,348",2,0,0,0,18,6,23,4,"2,126"
299,,Total,"1,770","3,853",9,1,5,3,65,17,43,6,"5,772"
300,TOTAL,Election,"358,827","608,357","5,207",292,912,239,"9,025","4,317","4,477",466,"992,119"


In [6]:
cols=[4,5, 6, 7, 8, 9, 10, 11,12]
voter_data_2024.drop(voter_data_2024.columns[cols], axis=1, inplace=True)
voter_data_2024

,0,1,2,3
0,Adair,Election \nDay,558,"1,915"
1,,Absentee,528,"1,001"
2,,Total,"1,086","2,916"
3,Adams,Election \nDay,266,878
4,,Absentee,310,639
...,...,...,...,...
296,,Total,"1,508","2,715"
297,Wright,Election \nDay,"1,045","2,505"
298,,Absentee,725,"1,348"
299,,Total,"1,770","3,853"


In [7]:
votes_2024_df=voter_data_2024.loc[voter_data_2024[1]=='Total'][[2, 3]].copy()
votes_2024_df=votes_2024_df.reset_index(drop=True)

In [8]:
votes_2024_df['county_id']=votes_2022_df['county_id']
votes_2024_df['NAME10']=votes_2022_df['NAME10']
votes_2024_df['og_district']=votes_2022_df['og_district']
votes_2024_df

,2,3,county_id,NAME10,og_district
0,"1,086","2,916",0,Adair,3
1,576,"1,517",1,Adams,3
2,"2,350","4,857",2,Allamakee,2
3,"1,686","4,704",3,Appanoose,3
4,970,"2,214",4,Audubon,4
...,...,...,...,...,...
94,"1,909","3,636",94,Winnebago,4
95,"5,321","6,427",95,Winneshiek,2
96,"16,145","25,969",96,Woodbury,4
97,"1,508","2,715",97,Worth,2


In [9]:
votes_2024_df = votes_2024_df.iloc[:, [2,3,0,1,4]]

In [10]:
votes_2024_df.columns=['county_id','NAME10', 'dem_votes', 'rep_votes', 'og_district']
votes_2024_df

,county_id,NAME10,dem_votes,rep_votes,og_district
0,0,Adair,"1,086","2,916",3
1,1,Adams,576,"1,517",3
2,2,Allamakee,"2,350","4,857",2
3,3,Appanoose,"1,686","4,704",3
4,4,Audubon,970,"2,214",4
...,...,...,...,...,...
94,94,Winnebago,"1,909","3,636",4
95,95,Winneshiek,"5,321","6,427",2
96,96,Woodbury,"16,145","25,969",4
97,97,Worth,"1,508","2,715",2


In [11]:
votes_2024_df=votes_2024_df.replace(',','',regex=True)

In [12]:
votes_2024_df['dem_votes']=votes_2024_df['dem_votes'].astype(int)
votes_2024_df['rep_votes']=votes_2024_df['rep_votes'].astype(int)
votes_2024_df['alt_district']=votes_2022_df['alt_district']
votes_2024_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   county_id     99 non-null     int64 
 1   NAME10        99 non-null     object
 2   dem_votes     99 non-null     int64 
 3   rep_votes     99 non-null     int64 
 4   og_district   99 non-null     int64 
 5   alt_district  99 non-null     int64 
dtypes: int64(5), object(1)
memory usage: 4.8+ KB


In [13]:
votes_2024_df

,county_id,NAME10,dem_votes,rep_votes,og_district,alt_district
0,0,Adair,1086,2916,3,4
1,1,Adams,576,1517,3,3
2,2,Allamakee,2350,4857,2,2
3,3,Appanoose,1686,4704,3,3
4,4,Audubon,970,2214,4,4
...,...,...,...,...,...,...
94,94,Winnebago,1909,3636,4,4
95,95,Winneshiek,5321,6427,2,2
96,96,Woodbury,16145,25969,4,4
97,97,Worth,1508,2715,2,4


In [14]:
votes_2024_df.to_csv('../data/voter_numbers_2024.csv', index=False,header=True)

In [15]:
df=pd.read_csv('../data/voter_numbers_2024.csv')
df

,county_id,NAME10,dem_votes,rep_votes,og_district,alt_district
0,0,Adair,1086,2916,3,4
1,1,Adams,576,1517,3,3
2,2,Allamakee,2350,4857,2,2
3,3,Appanoose,1686,4704,3,3
4,4,Audubon,970,2214,4,4
...,...,...,...,...,...,...
94,94,Winnebago,1909,3636,4,4
95,95,Winneshiek,5321,6427,2,2
96,96,Woodbury,16145,25969,4,4
97,97,Worth,1508,2715,2,4


Grab 2024 general election data

In [ ]:
raw_data_20=camelot.read_pdf('../data/canvsummary_2020.pdf', pages='2-15')
raw_data_20

In [ ]:
voter_data_2020=pd.concat([raw_data_20[i].df.tail(-1) for i in range(14)],ignore_index=True)
voter_data_2020.tail(5)

In [ ]:
cols_20=[4,5, 6, 7, 8, 9, 10, 11,12,13]
voter_data_2020.drop(voter_data_2020.columns[cols_20], axis=1, inplace=True)
voter_data_2020

In [ ]:
votes_2020_df=voter_data_2020.loc[voter_data_2020[1]=='Total'][[2, 3]].copy()
votes_2020_df=votes_2020_df.reset_index(drop=True)

In [ ]:
votes_2020_df['county_id']=votes_2022_df['county_id']
votes_2020_df['NAME10']=votes_2022_df['NAME10']
votes_2020_df['og_district']=votes_2022_df['og_district']
votes_2020_df

In [ ]:
votes_2020_df = votes_2020_df.iloc[:, [2,3,0,1,4]]

In [ ]:
votes_2020_df.columns=['county_id','NAME10', 'dem_votes', 'rep_votes', 'og_district']
votes_2020_df

In [ ]:
votes_2020_df=votes_2020_df.replace(',','',regex=True)
votes_2020_df['dem_votes']=votes_2020_df['dem_votes'].astype(int)
votes_2020_df['rep_votes']=votes_2020_df['rep_votes'].astype(int)
votes_2020_df.info()



In [ ]:
votes_2020_df.to_csv('../data/voter_numbers_2020.csv', index=False,header=True)